# VAZHI Clean DAPT Data Prep v2.0 — Own Sources Only

**Key changes from v1.1:**
1. **Own verified sources** — Sadhguru articles + classical lit (not external Sangraha)
2. **Tamil threshold >= 90%** (was 70% in v1.1 — too low, let in English/Tanglish)
3. **Chat replay 5-15%** of tokens — preserves instruction-following during DAPT
4. **~15M token budget** (was 55M in v1.1 — too much for 0.6B model)
5. **Incremental DAPT** on v5.3 SFT model (not vanilla)
6. **IndicAlign Wiki investigation** — check before committing

```
Step 1 (THIS NOTEBOOK): Data Prep — CPU only (Colab Pro)
  -> Output: CryptoYogi/vazhi-dapt-tamil-v2_0

Step 2: DAPT Training — GPU (Colab Pro)
  -> Input:  This dataset + CryptoYogi/vazhi-v5_3
  -> Output: CryptoYogi/vazhi-v5_3-dapt
```

**Runtime:** ~10-20 min on CPU

In [1]:
# Cell 1 — Dependencies
!pip install -q -U \
  "transformers>=4.45.0,<5.0.0" \
  "datasets>=2.21.0" \
  "huggingface_hub>=0.24.7"

print("\u2705 Dependencies installed (CPU-only)")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 111.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 52.5 MB/s eta 0:00:00
✅ Dependencies installed (CPU-only)


In [2]:
# Cell 2 — Configuration
import os
import re
import json
import random
import hashlib
import unicodedata
import numpy as np
from collections import Counter

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# === KEY CONFIG ===
HF_DATASET = "CryptoYogi/vazhi-dapt-tamil-v2_0"
MODEL_ID = "Qwen/Qwen3-0.6B"  # For tokenizer only
BLOCK_SIZE = 1024

# === DATA QUALITY FILTERS ===
TAMIL_THRESHOLD = 0.90  # >= 90% Tamil chars (up from 70% in v1.1)
MIN_CHARS = 100
MAX_CHARS = 50000
MAX_REPETITION_RATIO = 0.5

# === LOCAL FILE PATHS (uploaded to Colab) ===
SADHGURU_PATH = "articles_filtered_full.json"
DAPT_CORPUS_FILES = {
    "thirukkural": "37_thirukkural_corpus.json",
    "bharathiar": "40_bharathiar_corpus.json",
    "silapathikaram": "36_silapathikaram_corpus.json",
    "sangam": "38_sangam_corpus.json",
    "aathichoodi": "39_aathichoodi_corpus.json",
}
CHAT_REPLAY_FILES = [
    "conversational_fundamentals.json",
    "vazhi_behavior_pack.json",
]

print(f"\U0001f4cb DAPT Data Prep v2.0 Config:")
print(f"   Tokenizer:       {MODEL_ID}")
print(f"   Output:          {HF_DATASET}")
print(f"   Block size:      {BLOCK_SIZE} tokens")
print(f"   Tamil threshold: >= {TAMIL_THRESHOLD:.0%}")
print(f"   Sources:         Sadhguru articles + 5 classical lit + chat replay")
print(f"   Target:          ~15M tokens")

📋 DAPT Data Prep v2.0 Config:
   Tokenizer:       Qwen/Qwen3-0.6B
   Output:          CryptoYogi/vazhi-dapt-tamil-v2_0
   Block size:      1024 tokens
   Tamil threshold: >= 90%
   Sources:         Sadhguru articles + 5 classical lit + chat replay
   Target:          ~15M tokens


In [3]:
# Cell 3 — HuggingFace Login
from huggingface_hub import notebook_login
notebook_login()
print("\u2705 Logged in to HuggingFace")

✅ Logged in to HuggingFace


## Cell 4 — Upload Local Files

Upload these files from the project `data/sources/` directory to Colab:

**Sadhguru articles (primary source, ~87% of tokens):**
- `data/sources/sft/sadhguru-raw/articles_filtered_full.json`

**Classical literature (5 files):**
- `data/sources/dapt/36_silapathikaram_corpus.json`
- `data/sources/dapt/37_thirukkural_corpus.json`
- `data/sources/dapt/38_sangam_corpus.json`
- `data/sources/dapt/39_aathichoodi_corpus.json`
- `data/sources/dapt/40_bharathiar_corpus.json`

**Chat replay (instruction preservation):**
- `data/sources/sft/conversational_fundamentals.json`
- `data/sources/sft/vazhi_behavior_pack.json`

Use the Colab file upload button or `from google.colab import files; files.upload()`

In [4]:
# Cell 5 — Load Tokenizer
from transformers import AutoTokenizer

print(f"\U0001f4e5 Loading tokenizer from {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"\u2705 Tokenizer ready: {len(tokenizer)} tokens")
print(f"   eos_token: {tokenizer.eos_token!r} (ID {tokenizer.eos_token_id})")

📥 Loading tokenizer from Qwen/Qwen3-0.6B...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

✅ Tokenizer ready: 151669 tokens
   eos_token: '<|im_end|>' (ID 151645)


In [17]:
# Cell 6 — Cleaning & Quality Functions
# Reused from v1.1 + create_sadhguru_qa_v2.py

# Zero-width and invisible characters to strip
_INVISIBLE_RE = re.compile(
    '['
    '\u200b'  # zero-width space
    '\u200c'  # zero-width non-joiner
    '\u200d'  # zero-width joiner
    '\u200e'  # left-to-right mark
    '\u200f'  # right-to-left mark
    '\u00ad'  # soft hyphen
    '\ufeff'  # byte order mark
    '\u2060'  # word joiner
    '\u2061'  # function application
    '\u2062'  # invisible times
    '\u2063'  # invisible separator
    '\u2064'  # invisible plus
    '\ufff9'  # interlinear annotation anchor
    '\ufffa'  # interlinear annotation separator
    '\ufffb'  # interlinear annotation terminator
    ']'
)

# Control characters (keep newline \n, tab \t, carriage return \r)
_CONTROL_RE = re.compile(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f-\x9f]')


def nfkc_normalize(text):
    """NFKC normalize + strip invisible/control chars + collapse whitespace."""
    text = unicodedata.normalize('NFKC', text)
    text = text.replace('\ufffd', '')
    text = _INVISIBLE_RE.sub('', text)
    text = _CONTROL_RE.sub('', text)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r' *\n *', '\n', text)
    return text.strip()


def clean_article_text(text):
    """Remove HTML/markdown artifacts from scraped Sadhguru article text.
    Regex patterns from scripts/create_sadhguru_qa_v2.py:25-60.
    """
    # Remove SadhguruImage tags
    text = re.sub(r'\[/?[Ss]adhguru[Ii]mage[^\]]*\]', '', text)
    # Remove pullquote tags (keep content)
    text = re.sub(r'\[/?[Pp]ull[Qq]uote\s*\]', '', text)
    # Remove separator tags
    text = re.sub(r'\[[Ss]eparator[^\]]*\]', '', text)
    # Remove photocredit tags
    text = re.sub(r'\[/?[Pp]hoto[Cc]redit[^\]]*\]', '', text)
    # Remove any remaining bracket tags
    text = re.sub(r'\[/?[A-Za-z]+[^\]]*\]', '', text)
    # Remove footnotes section
    text = re.sub(r'\n\s*\u0b95\u0bc1\u0bb1\u0bbf\u0baa\u0bcd\u0baa\u0bc1\s*:.*$', '', text, flags=re.DOTALL)
    # Remove cross-references
    text = re.sub(r'\u0bae\u0bc7\u0bb2\u0bc1\u0bae\u0bcd\s+\u0baa\u0bb2\s+\u0b95\u0ba4\u0bc8\u0b95\u0bb3\u0bc8\u0b95\u0bcd\s+\u0b95\u0bbe\u0ba3\u0bcd\u0b95.*$', '', text, flags=re.DOTALL)
    # Remove URLs
    text = re.sub(r'https?://\S+', '', text)
    # Remove markdown links (keep text)
    text = re.sub(r'\[([^\]]+)\]\([^\)]+\)', r'\1', text)
    # Remove single-quoted lines
    text = re.sub(r"^'[^']+' *$", '', text, flags=re.MULTILINE)
    # Clean whitespace
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'  +', ' ', text)
    return text.strip()


def tamil_char_pct(text):
      """Compute Tamil Unicode char % (among non-whitespace, non-digit chars)."""
      if not text:
          return 0.0
      tamil = sum(1 for c in text if '\u0B80' <= c <= '\u0BFF')
      total = sum(1 for c in text if not c.isspace() and not c.isdigit())
      return tamil / total if total > 0 else 0.0



def has_excessive_repetition(text, threshold=MAX_REPETITION_RATIO):
    """Detect repeated lines (boilerplate)."""
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    if len(lines) < 3:
        return False
    line_counts = Counter(lines)
    most_common_count = line_counts.most_common(1)[0][1]
    return most_common_count / len(lines) > threshold


def text_hash(text):
    """MD5 hash for dedup."""
    return hashlib.md5(text.encode('utf-8')).hexdigest()


def is_quality_tamil(text):
    """Combined quality gate for DAPT text."""
    if len(text) < MIN_CHARS:
        return False, 'too_short'
    if len(text) > MAX_CHARS:
        return False, 'too_long'
    pct = tamil_char_pct(text)
    if pct < TAMIL_THRESHOLD:
        return False, f'low_tamil_{pct:.0%}'
    if has_excessive_repetition(text):
        return False, 'repetitive'
    return True, 'ok'


# Quick test
test_text = "  \u200b\u0ba4\u0bae\u0bbf\u0bb4\u0bcd\ufffd  \u0ba8\u0bbe\u0b9f\u0bc1  \n\n\n\n  test  "
cleaned = nfkc_normalize(test_text)
print(f"\u2705 Cleaning pipeline ready")
print(f"   Test: {test_text!r}")
print(f"   Clean: {cleaned!r}")
assert '\ufffd' not in cleaned
assert '\u200b' not in cleaned
assert '\n\n\n' not in cleaned
print(f"   All assertions passed")

✅ Cleaning pipeline ready
   Test: '  \u200bதமிழ்�  நாடு  \n\n\n\n  test  '
   Clean: 'தமிழ் நாடு\n\ntest'
   All assertions passed


In [6]:
# Cell 7 — IndicAlign Wiki_Chat/Wiki_Conv Investigation
# Quick quality check on streaming IndicAlign subsets
# Decision gate: include if avg Tamil >= 90%, else skip

from datasets import load_dataset

WIKI_SUBSETS = [
    {"config": "Wiki_Chat", "split": "tam", "label": "Wiki_Chat"},
    {"config": "Wiki_Conv", "split": "tam", "label": "Wiki_Conv"},
]

wiki_reports = {}

for subset in WIKI_SUBSETS:
    label = subset['label']
    print(f"\n\U0001f50d Investigating {label}...")

    try:
        ds_stream = load_dataset(
            "ai4bharat/IndicAlign", subset['config'],
            split=subset['split'], streaming=True
        )
    except Exception as e:
        print(f"   \u26a0\ufe0f Failed to load {label}: {e}")
        wiki_reports[label] = {'status': 'failed', 'error': str(e)}
        continue

    samples = []
    tamil_pcts = []
    lengths = []

    for i, item in enumerate(ds_stream):
        if i >= 50:
            break
        # IndicAlign has varying column names — check common ones
        text = item.get('text', item.get('content', item.get('response', '')))
        if not text:
            # Try all string values
            for v in item.values():
                if isinstance(v, str) and len(v) > 50:
                    text = v
                    break
        if not text:
            continue

        text = nfkc_normalize(text)
        pct = tamil_char_pct(text)
        tamil_pcts.append(pct)
        lengths.append(len(text))
        if len(samples) < 5:
            samples.append((pct, text[:200]))

    if not tamil_pcts:
        print(f"   \u274c No text found. Columns: {list(item.keys()) if 'item' in dir() else 'N/A'}")
        wiki_reports[label] = {'status': 'no_text'}
        continue

    avg_tamil = np.mean(tamil_pcts)
    avg_len = np.mean(lengths)
    above_90 = sum(1 for p in tamil_pcts if p >= 0.90)

    print(f"   Samples checked: {len(tamil_pcts)}")
    print(f"   Avg Tamil:       {avg_tamil:.1%}")
    print(f"   Above 90%:       {above_90}/{len(tamil_pcts)}")
    print(f"   Avg length:      {avg_len:.0f} chars")

    recommendation = "GO" if avg_tamil >= 0.85 and above_90 > len(tamil_pcts) * 0.5 else "SKIP"
    print(f"   \U0001f449 Recommendation: {recommendation}")

    wiki_reports[label] = {
        'status': 'checked',
        'avg_tamil': avg_tamil,
        'above_90_pct': above_90 / len(tamil_pcts),
        'avg_len': avg_len,
        'recommendation': recommendation,
    }

    print(f"\n   Sample texts:")
    for pct, text in samples[:3]:
        print(f"   [{pct:.0%}] {text[:150]}...")

print(f"\n{'='*60}")
print(f"\U0001f4cb WIKI INVESTIGATION SUMMARY")
print(f"{'='*60}")
for label, report in wiki_reports.items():
    if report['status'] == 'checked':
        print(f"   {label}: avg Tamil {report['avg_tamil']:.1%}, "
              f"above 90% = {report['above_90_pct']:.0%} -> {report['recommendation']}")
    else:
        print(f"   {label}: {report['status']}")
print(f"\n\u2b07\ufe0f Set INCLUDE_WIKI in next cell based on these results.")


🔍 Investigating Wiki_Chat...
   ⚠️ Failed to load Wiki_Chat: Dataset 'ai4bharat/IndicAlign' doesn't exist on the Hub or cannot be accessed.

🔍 Investigating Wiki_Conv...
   ⚠️ Failed to load Wiki_Conv: Dataset 'ai4bharat/IndicAlign' doesn't exist on the Hub or cannot be accessed.

📋 WIKI INVESTIGATION SUMMARY
   Wiki_Chat: failed
   Wiki_Conv: failed

⬇️ Set INCLUDE_WIKI in next cell based on these results.


In [7]:
# Cell 8 — Conditional Wiki Processing
# User sets True/False based on Cell 7 investigation results

INCLUDE_WIKI = False  # Set True if Cell 7 showed GO recommendation

wiki_texts = []
wiki_stats = {'kept': 0, 'dropped': 0, 'tokens': 0}

if INCLUDE_WIKI:
    seen_hashes = set()
    for subset in WIKI_SUBSETS:
        label = subset['label']
        report = wiki_reports.get(label, {})
        if report.get('recommendation') != 'GO':
            print(f"   Skipping {label} (recommendation was not GO)")
            continue

        print(f"\U0001f4e5 Processing {label}...")
        ds_stream = load_dataset(
            "ai4bharat/IndicAlign", subset['config'],
            split=subset['split'], streaming=True
        )

        for item in ds_stream:
            text = item.get('text', item.get('content', item.get('response', '')))
            if not text:
                for v in item.values():
                    if isinstance(v, str) and len(v) > 50:
                        text = v
                        break
            if not text:
                continue

            text = nfkc_normalize(text)
            ok, reason = is_quality_tamil(text)
            if not ok:
                wiki_stats['dropped'] += 1
                continue

            h = text_hash(text)
            if h in seen_hashes:
                wiki_stats['dropped'] += 1
                continue
            seen_hashes.add(h)

            wiki_texts.append(text)
            wiki_stats['kept'] += 1

        print(f"   \u2705 {label}: kept {wiki_stats['kept']}")

    if wiki_texts:
        sample_tokens = sum(len(tokenizer.encode(t, add_special_tokens=False)) for t in wiki_texts[:100])
        est_tokens = int(sample_tokens / min(len(wiki_texts), 100) * len(wiki_texts))
        wiki_stats['tokens'] = est_tokens
        print(f"   Total Wiki: {len(wiki_texts)} docs, ~{est_tokens:,} est. tokens")
else:
    print("\u23ed\ufe0f Skipping Wiki processing (INCLUDE_WIKI = False)")
    print("   Reason: Using own verified sources only for data quality control")

⏭️ Skipping Wiki processing (INCLUDE_WIKI = False)
   Reason: Using own verified sources only for data quality control


In [13]:
 # Cell 4 — Download Source Files from HuggingFace
from huggingface_hub import hf_hub_download
import os

SOURCES_REPO = "CryptoYogi/vazhi-dapt-sources-v2_0"

FILES = [
      "articles_filtered_full.json",
      "36_silapathikaram_corpus.json",
      "37_thirukkural_corpus.json",
      "38_sangam_corpus.json",
      "39_aathichoodi_corpus.json",
      "40_bharathiar_corpus.json",
      "conversational_fundamentals.json",
      "vazhi_behavior_pack.json",
  ]

print(f"📥 Downloading {len(FILES)} source files from {SOURCES_REPO}...")
for fname in FILES:
      if os.path.exists(fname):
          print(f"   ✅ {fname} (already present)")
          continue
      path = hf_hub_download(
          repo_id=SOURCES_REPO, filename=fname,
          repo_type="dataset", local_dir=".",
      )
      size = os.path.getsize(fname)
      print(f"   ✅ {fname} ({size:,} bytes)")

print(f"\n✅ All {len(FILES)} source files ready")


📥 Downloading 8 source files from CryptoYogi/vazhi-dapt-sources-v2_0...
   ✅ articles_filtered_full.json (already present)


36_silapathikaram_corpus.json: 0.00B [00:00, ?B/s]

   ✅ 36_silapathikaram_corpus.json (86,093 bytes)


37_thirukkural_corpus.json: 0.00B [00:00, ?B/s]

   ✅ 37_thirukkural_corpus.json (966,431 bytes)


...

   ✅ 40_bharathiar_corpus.json (543,825 bytes)


conversational_fundamentals.json: 0.00B [00:00, ?B/s]

   ✅ conversational_fundamentals.json (120,082 bytes)


vazhi_behavior_pack.json: 0.00B [00:00, ?B/s]

   ✅ vazhi_behavior_pack.json (64,764 bytes)

✅ All 8 source files ready


In [20]:
# Cell 9 — Process Sadhguru Articles (Primary Source, ~87% of tokens)

print(f"\U0001f4e5 Loading Sadhguru articles from {SADHGURU_PATH}...")
with open(SADHGURU_PATH, 'r', encoding='utf-8') as f:
    articles = json.load(f)
print(f"   Loaded {len(articles)} articles")

sadhguru_texts = []
sadhguru_stats = {'kept': 0, 'dropped_short': 0, 'dropped_tamil': 0,
                  'dropped_repetition': 0, 'dropped_dedup': 0}
seen_hashes_sg = set()

for article in articles:
    raw_text = article.get('tamil_text', '')
    if not raw_text:
        sadhguru_stats['dropped_short'] += 1
        continue

    # Clean article artifacts first, then NFKC normalize
    text = clean_article_text(raw_text)
    text = nfkc_normalize(text)

    # Quality gate
    if len(text) < MIN_CHARS:
        sadhguru_stats['dropped_short'] += 1
        continue

    pct = tamil_char_pct(text)
    if pct < TAMIL_THRESHOLD:
        sadhguru_stats['dropped_tamil'] += 1
        continue

    if has_excessive_repetition(text):
        sadhguru_stats['dropped_repetition'] += 1
        continue

    h = text_hash(text)
    if h in seen_hashes_sg:
        sadhguru_stats['dropped_dedup'] += 1
        continue
    seen_hashes_sg.add(h)

    sadhguru_texts.append(text)
    sadhguru_stats['kept'] += 1

# Compute token count on sample, then estimate total
sample_size = min(50, len(sadhguru_texts))
sample_tokens = sum(len(tokenizer.encode(t, add_special_tokens=False)) for t in sadhguru_texts[:sample_size])
avg_tokens_per_doc = sample_tokens / sample_size
est_sadhguru_tokens = int(avg_tokens_per_doc * len(sadhguru_texts))

print(f"\n\u2705 Sadhguru articles processed:")
print(f"   Kept:             {sadhguru_stats['kept']}")
print(f"   Dropped (short):  {sadhguru_stats['dropped_short']}")
print(f"   Dropped (tamil):  {sadhguru_stats['dropped_tamil']}")
print(f"   Dropped (repeat): {sadhguru_stats['dropped_repetition']}")
print(f"   Dropped (dedup):  {sadhguru_stats['dropped_dedup']}")
print(f"   Avg tokens/doc:   {avg_tokens_per_doc:.0f}")
print(f"   Est. total tokens: {est_sadhguru_tokens:,}")

# Spot check
if sadhguru_texts:
    sample = random.choice(sadhguru_texts)
    print(f"\n\U0001f4d6 Sample (Tamil {tamil_char_pct(sample):.0%}, {len(sample)} chars):")
    print(f"   {sample[:300]}...")

📥 Loading Sadhguru articles from articles_filtered_full.json...
   Loaded 562 articles

✅ Sadhguru articles processed:
   Kept:             561
   Dropped (short):  0
   Dropped (tamil):  1
   Dropped (repeat): 0
   Dropped (dedup):  0
   Avg tokens/doc:   6912
   Est. total tokens: 3,877,822

📖 Sample (Tamil 98%, 10420 chars):
   சகாதேவன் மட்டும் இதில் மயங்காமல் துரியோதனனிடம் வழக்கமான தன் இடைவெளியை தொடர்ந்தான்.

இப்படித்தான் வீரமான, பயமறியாத, வெட்டு ஒன்று துண்டு இரண்டு எனும் வழி சென்று கொண்டிருந்த துரியோதனனுக்குள் வஞ்சகம் புகட்டப்பட்டது. எப்போதுமே பொறாமையும், வெறுப்பும், ஆத்திரமும் மண்டிக்கிடந்த துரியோதனின் நெஞ்சுக்குள் வஞ்ச...


In [21]:
# Cell 10 — Process Classical Literature
# Format-specific extraction for 5 DAPT corpus files

classical_texts = []
classical_stats = {}


def process_thirukkural(data):
    """Thirukkural: concatenate verse + meaning."""
    texts = []
    for item in data:
        verse = item.get('tamil', '')
        meaning = item.get('meaning_tamil', '')
        if verse and meaning:
            texts.append(f"{verse} {meaning}")
    return texts


def process_bharathiar(data):
    """Bharathiar: extract full_text from poems list."""
    texts = []
    poems = data.get('poems', [])
    for poem in poems:
        text = poem.get('full_text', '')
        if text:
            texts.append(text)
    return texts


def process_silapathikaram(data):
    """Silapathikaram: extract text field."""
    return [item.get('text', '') for item in data if item.get('text')]


def process_sangam(data):
    """Sangam: extract text field."""
    return [item.get('text', '') for item in data if item.get('text')]


def process_aathichoodi(data):
    """Aathichoodi: extract verse + meaning from both aathichoodi and konrai_venthan."""
    texts = []
    for section_key in ['aathichoodi', 'konrai_venthan']:
        section = data.get(section_key, {})
        verses = section.get('verses', [])
        for v in verses:
            verse = v.get('verse', '')
            meaning = v.get('meaning_tamil', '')
            if verse and meaning:
                texts.append(f"{verse} - {meaning}")
            elif verse:
                texts.append(verse)
    return texts


PROCESSORS = {
    'thirukkural': process_thirukkural,
    'bharathiar': process_bharathiar,
    'silapathikaram': process_silapathikaram,
    'sangam': process_sangam,
    'aathichoodi': process_aathichoodi,
}

for name, filename in DAPT_CORPUS_FILES.items():
    print(f"\U0001f4e5 Processing {name} ({filename})...")

    if not os.path.exists(filename):
        print(f"   \u26a0\ufe0f File not found: {filename}. Skipping.")
        classical_stats[name] = {'kept': 0, 'dropped': 0, 'error': 'file_not_found'}
        continue

    with open(filename, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # Extract raw texts using format-specific processor
    processor = PROCESSORS[name]
    raw_texts = processor(data)
    print(f"   Extracted {len(raw_texts)} raw texts")

    # Apply quality gate
    kept = 0
    dropped = 0
    for text in raw_texts:
        text = nfkc_normalize(text)
        if not text or len(text) < 10:  # Lower min for short classical texts
            dropped += 1
            continue
        # Classical lit gets relaxed Tamil threshold since it's all Tamil by nature
        pct = tamil_char_pct(text)
        if pct < 0.80:  # Slightly relaxed for classical (spaces, punctuation)
            dropped += 1
            continue
        classical_texts.append(text)
        kept += 1

    classical_stats[name] = {'kept': kept, 'dropped': dropped}
    print(f"   \u2705 {name}: kept {kept}, dropped {dropped}")

# Estimate classical tokens
if classical_texts:
    sample_size = min(100, len(classical_texts))
    sample_tokens = sum(len(tokenizer.encode(t, add_special_tokens=False)) for t in classical_texts[:sample_size])
    avg_tokens = sample_tokens / sample_size
    est_classical_tokens = int(avg_tokens * len(classical_texts))
else:
    est_classical_tokens = 0

print(f"\n\U0001f4ca Classical literature summary:")
print(f"   Total texts: {len(classical_texts)}")
print(f"   Est. tokens: {est_classical_tokens:,}")
for name, stats in classical_stats.items():
    print(f"   {name}: kept {stats['kept']}, dropped {stats['dropped']}")

📥 Processing thirukkural (37_thirukkural_corpus.json)...
   Extracted 1330 raw texts
   ✅ thirukkural: kept 1330, dropped 0
📥 Processing bharathiar (40_bharathiar_corpus.json)...
   Extracted 109 raw texts
   ✅ bharathiar: kept 109, dropped 0
📥 Processing silapathikaram (36_silapathikaram_corpus.json)...
   Extracted 142 raw texts
   ✅ silapathikaram: kept 142, dropped 0
📥 Processing sangam (38_sangam_corpus.json)...
   Extracted 63 raw texts
   ✅ sangam: kept 63, dropped 0
📥 Processing aathichoodi (39_aathichoodi_corpus.json)...
   Extracted 200 raw texts
   ✅ aathichoodi: kept 200, dropped 0

📊 Classical literature summary:
   Total texts: 1844
   Est. tokens: 359,635
   thirukkural: kept 1330, dropped 0
   bharathiar: kept 109, dropped 0
   silapathikaram: kept 142, dropped 0
   sangam: kept 63, dropped 0
   aathichoodi: kept 200, dropped 0


In [19]:
  # Debug: check Sadhguru file and first 5 articles
  import json, os

  print(f"File size: {os.path.getsize(SADHGURU_PATH):,} bytes")
  with open(SADHGURU_PATH, 'r', encoding='utf-8') as f:
      articles = json.load(f)
  print(f"Articles loaded: {len(articles)}")
  print(f"Type: {type(articles)}")
  if articles:
      print(f"First item keys: {list(articles[0].keys()) if isinstance(articles[0], dict) else type(articles[0])}")

  for i, article in enumerate(articles[:5]):
      raw = article.get('tamil_text', '')
      cleaned = clean_article_text(raw)
      normalized = nfkc_normalize(cleaned)
      pct = tamil_char_pct(normalized)
      print(f"\nArticle {i}: raw={len(raw):,} chars, cleaned={len(cleaned):,}, tamil={pct:.1%}")
      print(f"  First 100 chars: {normalized[:100]}")


File size: 10,890,888 bytes
Articles loaded: 562
Type: <class 'list'>
First item keys: ['url', 'title', 'category', 'tamil_text', 'char_count', 'word_count', 'tamil_char_count']

Article 0: raw=5,484 chars, cleaned=5,445, tamil=96.8%
  First 100 chars: ஸ்வதர்மம் என்று அவர் சொல்லும்போது, ஸ்வ என்றால் சுயம்,தர்மம்என்றால் சட்டம். அதாவது சுய சட்டம் அல்லது 

Article 1: raw=9,378 chars, cleaned=9,251, tamil=96.5%
  First 100 chars: தர்மத்தை நிலைநாட்ட வேண்டும் என்ற விருப்பத்துடன் கிருஷ்ணர் தன் வாழ்நாள் முழுவதையும் கழித்தார். பல விஷ

Article 2: raw=4,718 chars, cleaned=4,679, tamil=97.5%
  First 100 chars: கேள்வியாளர்:காத்திருப்பது எவ்வளவு முக்கியம் என்று நீங்கள் பேசிக் கொண்டிருந்தீர்கள். நான் கேட்க விரும

Article 3: raw=7,317 chars, cleaned=7,253, tamil=96.6%
  First 100 chars: கேள்வி:சத்குரு, எங்கள் பள்ளி, கல்லூரி காலம் முழுவதும், தன்னையே நேசிப்பது, தனக்கே முக்கியத்துவம் கொடு

Article 4: raw=2,308 chars, cleaned=2,145, tamil=96.3%
  First 100 chars: பொங்கல் திருவிழா குறித்த சத்குருவின் வாசகங்

In [23]:
# Cell 11 — Quality Verification Gate

# Compute Tamil % on all prose texts (excluding chat replay)
all_prose_texts = sadhguru_texts + classical_texts + wiki_texts

sample_size = min(500, len(all_prose_texts))
sample_indices = random.sample(range(len(all_prose_texts)), sample_size)
sample_pcts = [tamil_char_pct(all_prose_texts[i]) for i in sample_indices]
avg_tamil = np.mean(sample_pcts)

# Estimate total tokens for prose
est_total_prose_tokens = est_sadhguru_tokens + est_classical_tokens + wiki_stats.get('tokens', 0)

print(f"{'='*60}")
print(f"\U0001f4ca QUALITY VERIFICATION GATE")
print(f"{'='*60}")
print(f"")
print(f"{'Source':<20} {'Docs':>8} {'Est. Tokens':>14}")
print(f"{'-'*45}")
print(f"{'Sadhguru articles':<20} {len(sadhguru_texts):>8,} {est_sadhguru_tokens:>14,}")
print(f"{'Classical lit':<20} {len(classical_texts):>8,} {est_classical_tokens:>14,}")
if wiki_texts:
    print(f"{'Wiki (IndicAlign)':<20} {len(wiki_texts):>8,} {wiki_stats.get('tokens', 0):>14,}")
print(f"{'-'*45}")
print(f"{'TOTAL PROSE':<20} {len(all_prose_texts):>8,} {est_total_prose_tokens:>14,}")
print(f"")
print(f"   Avg Tamil (sample): {avg_tamil:.1%}")
print(f"   Min Tamil (sample): {min(sample_pcts):.1%}")
print(f"   Median Tamil:       {np.median(sample_pcts):.1%}")
print(f"")

# Fail-fast checks
assert avg_tamil >= 0.90, f"STOP: Average Tamil {avg_tamil:.1%} < 90%"
print(f"\u2705 Average Tamil >= 90% check passed")

assert est_total_prose_tokens >= 4_000_000, (
    f"STOP: Only {est_total_prose_tokens:,} estimated tokens (need >= 5M)"
)
print(f"\u2705 Token budget >= 5M check passed")

print(f"\n\u2705 All quality gates passed. Proceeding to chat replay + packing.")

📊 QUALITY VERIFICATION GATE

Source                   Docs    Est. Tokens
---------------------------------------------
Sadhguru articles         561      3,877,822
Classical lit           1,844        359,635
---------------------------------------------
TOTAL PROSE             2,405      4,237,457

   Avg Tamil (sample): 97.8%
   Min Tamil (sample): 91.5%
   Median Tamil:       97.9%

✅ Average Tamil >= 90% check passed
✅ Token budget >= 5M check passed

✅ All quality gates passed. Proceeding to chat replay + packing.


In [24]:
# Cell 12 — Chat Replay Data (Instruction Preservation)
# Format as ChatML to preserve instruction-following during DAPT
# Target: 5-15% of total tokens

chat_texts = []
chat_stats = {}

for chat_file in CHAT_REPLAY_FILES:
    if not os.path.exists(chat_file):
        print(f"   \u26a0\ufe0f {chat_file} not found. Skipping.")
        continue

    with open(chat_file, 'r', encoding='utf-8') as f:
        data = json.load(f)

    source_name = os.path.splitext(chat_file)[0]
    kept = 0

    for item in data:
        instruction = item.get('instruction', '')
        output = item.get('output', '')
        if not instruction or not output:
            continue

        # Format as ChatML
        chatml = (
            f"<|im_start|>user\n{instruction}<|im_end|>\n"
            f"<|im_start|>assistant\n{output}<|im_end|>"
        )
        chat_texts.append(chatml)
        kept += 1

    chat_stats[source_name] = kept
    print(f"   {source_name}: {kept} samples formatted as ChatML")

# Compute chat replay tokens
if chat_texts:
    chat_token_count = sum(len(tokenizer.encode(t, add_special_tokens=False)) for t in chat_texts)
    chat_pct_of_total = chat_token_count / (est_total_prose_tokens + chat_token_count)
else:
    chat_token_count = 0
    chat_pct_of_total = 0

print(f"\n\U0001f4ca Chat replay summary:")
print(f"   Total samples:  {len(chat_texts)}")
print(f"   Total tokens:   {chat_token_count:,}")
print(f"   % of dataset:   {chat_pct_of_total:.1%}")

if chat_pct_of_total < 0.01:
    print(f"   \u26a0\ufe0f Chat replay < 1% — instruction preservation may be weak")
elif chat_pct_of_total > 0.20:
    print(f"   \u26a0\ufe0f Chat replay > 20% — may dilute Tamil language learning")
else:
    print(f"   \u2705 Chat replay in target range (1-20%)")

# Show a sample
if chat_texts:
    print(f"\n\U0001f4d6 Sample ChatML:")
    print(f"   {chat_texts[0][:200]}...")

   conversational_fundamentals: 268 samples formatted as ChatML
   vazhi_behavior_pack: 116 samples formatted as ChatML

📊 Chat replay summary:
   Total samples:  384
   Total tokens:   59,259
   % of dataset:   1.4%
   ✅ Chat replay in target range (1-20%)

📖 Sample ChatML:
   <|im_start|>user
வணக்கம்<|im_end|>
<|im_start|>assistant
வணக்கம்! நான் வழி, உங்கள் AI உதவியாளர். உங்களுக்கு எப்படி உதவ வேண்டும்?<|im_end|>...


In [25]:
# Cell 13 — Combine & Shuffle All Sources

all_texts = sadhguru_texts + classical_texts + wiki_texts + chat_texts

# Shuffle with fixed seed for reproducibility
random.seed(RANDOM_SEED)
random.shuffle(all_texts)

total_chars = sum(len(t) for t in all_texts)

print(f"\U0001f4ca Combined corpus:")
print(f"   Total documents:  {len(all_texts):,}")
print(f"   Total characters: {total_chars:,}")
print(f"   Breakdown:")
print(f"     Sadhguru:     {len(sadhguru_texts):,} docs")
print(f"     Classical:    {len(classical_texts):,} docs")
print(f"     Wiki:         {len(wiki_texts):,} docs")
print(f"     Chat replay:  {len(chat_texts):,} docs")

📊 Combined corpus:
   Total documents:  2,789
   Total characters: 4,380,164
   Breakdown:
     Sadhguru:     561 docs
     Classical:    1,844 docs
     Wiki:         0 docs
     Chat replay:  384 docs


In [26]:
# Cell 14 — Pack into 1024-Token Blocks
# Reuses v1.1 packing strategy: concatenate with EOS separator, split into blocks

print(f"\U0001f4e6 Packing {len(all_texts):,} docs into {BLOCK_SIZE}-token blocks...")

all_token_ids = []
eos_id = tokenizer.eos_token_id

for i, text in enumerate(all_texts):
    tokens = tokenizer.encode(text, add_special_tokens=False)
    all_token_ids.extend(tokens)
    all_token_ids.append(eos_id)

    if (i + 1) % 500 == 0:
        print(f"   ...tokenized {i + 1:,}/{len(all_texts):,} docs")

print(f"   Total token stream: {len(all_token_ids):,} tokens")

# Split into fixed-length blocks
n_blocks = len(all_token_ids) // BLOCK_SIZE
trimmed = all_token_ids[:n_blocks * BLOCK_SIZE]
blocks = [trimmed[i * BLOCK_SIZE:(i + 1) * BLOCK_SIZE] for i in range(n_blocks)]

total_tokens = len(blocks) * BLOCK_SIZE
discarded = len(all_token_ids) - len(trimmed)

print(f"\n\u2705 Packed into {len(blocks):,} blocks of {BLOCK_SIZE} tokens")
print(f"   Total training tokens: {total_tokens:,}")
print(f"   Discarded tail:        {discarded:,} tokens")
print(f"   Efficiency:            {total_tokens / len(all_token_ids):.1%}")

📦 Packing 2,789 docs into 1024-token blocks...
   ...tokenized 500/2,789 docs
   ...tokenized 1,000/2,789 docs
   ...tokenized 1,500/2,789 docs
   ...tokenized 2,000/2,789 docs
   ...tokenized 2,500/2,789 docs
   Total token stream: 4,795,785 tokens

✅ Packed into 4,683 blocks of 1024 tokens
   Total training tokens: 4,795,392
   Discarded tail:        393 tokens
   Efficiency:            100.0%


In [27]:
# Cell 15 — Block Quality Verification

print(f"\U0001f50d Verifying block quality (10 random samples)...\n")

random.seed(RANDOM_SEED)
check_indices = random.sample(range(len(blocks)), min(10, len(blocks)))
block_tamil_pcts = []

for idx in check_indices:
    decoded = tokenizer.decode(blocks[idx])
    pct = tamil_char_pct(decoded)
    block_tamil_pcts.append(pct)
    # Chat replay blocks will have lower Tamil % (expected)
    is_chat = '<|im_start|>' in decoded
    tag = " [chat]" if is_chat else ""
    print(f"   Block {idx:>5}: Tamil {pct:.0%}{tag} | {decoded[:120]}...")

# Full distribution (sample 200 blocks)
print(f"\n\U0001f4ca Tamil % distribution across {min(200, len(blocks))} sampled blocks:")
sample_block_indices = random.sample(range(len(blocks)), min(200, len(blocks)))
all_block_pcts = []
for idx in sample_block_indices:
    decoded = tokenizer.decode(blocks[idx])
    all_block_pcts.append(tamil_char_pct(decoded))

pct_array = np.array(all_block_pcts)
buckets = [(0, 0.5), (0.5, 0.7), (0.7, 0.8), (0.8, 0.9), (0.9, 1.01)]
for lo, hi in buckets:
    count = np.sum((pct_array >= lo) & (pct_array < hi))
    bar = '#' * int(count / len(pct_array) * 40)
    print(f"   {lo:.0%}-{hi:.0%}: {count:>4} ({count/len(pct_array):>5.1%}) {bar}")

print(f"\n   Mean:   {np.mean(all_block_pcts):.1%}")
print(f"   Median: {np.median(all_block_pcts):.1%}")
print(f"   Min:    {np.min(all_block_pcts):.1%}")

# Blocks below 50% are concerning unless they're chat replay
low_blocks = np.sum(pct_array < 0.50)
if low_blocks > len(pct_array) * 0.15:
    print(f"   \u26a0\ufe0f {low_blocks} blocks < 50% Tamil — check if too much chat replay")
else:
    print(f"   \u2705 Block quality looks good")

🔍 Verifying block quality (10 random samples)...

   Block   912: Tamil 96% |  இருந்தாலும் எதிர்மறையாக இருந்தாலும், அது எதிர்ப்பும் இடர்ப்பாடும் இல்லாமல் நடக்கமுடியாது. ஒவ்வொரு சின்னச்சின்ன விஷயத்தி...
   Block   204: Tamil 98% | �ன ஒலி அமைப்புகளை கொண்ட அறிவியல் பூர்வமான மொழியை உருவாக்கும் சூழல் நமக்கு இந்தியாவில் இருந்தது. இந்த கலாச்சாரத்தில் நாம்...
   Block  2253: Tamil 97% | ் சாப்பிட்ட உடனேயே தூங்கவேண்டாம்

இந்த விதமான மனநிலையில் சிலர் இருந்துகொண்டிருக்கின்றனர். அதாவது, உணவினால் தங்கள் வயிற்ற...
   Block  2006: Tamil 93% | <|im_end|>வினைக்கண் வினையுடையான் கேண்மைவே றாக நினைப்பானை நீங்கும் திரு. மேற்க்கொண்ட தொழிலில் எப்போதும் முயற்சி உடையவனின்...
   Block  1828: Tamil 77% [chat] |  அன்பு மற்றும் உதவியை வழங்கினால் போதும். ஒவ்வொரு மனிதரிடமும் ஒரு தனித்துவமான சாத்தியத்திற்கான திறமை இருக்கிறது.<|im_end|...
   Block  1143: Tamil 88% [chat] | றிக்கோளும் இல்லை, நான் வெறுமனே விளையாடிக் கொண்டிருக்கிறேன்" என்று சொல்லும்போது, நான் லேசாக எடுத்துக் கொள்வதாக அவர்கள் நி...
   Block   839: Tamil 92

In [28]:
# Cell 16 — Create HuggingFace Dataset & Upload

from datasets import Dataset
from huggingface_hub import HfApi

# Create dataset with input_ids column (no train/eval split for DAPT)
packed_dataset = Dataset.from_dict({
    "input_ids": blocks,
    "attention_mask": [[1] * BLOCK_SIZE for _ in blocks],
    "labels": [list(b) for b in blocks],
})

print(f"\U0001f4ca Dataset created:")
print(f"   Blocks:       {len(packed_dataset):,}")
print(f"   Total tokens: {len(packed_dataset) * BLOCK_SIZE:,}")
print(f"   Columns:      {packed_dataset.column_names}")

# Upload
print(f"\n\U0001f4e4 Uploading to {HF_DATASET}...")

# Build source summary
sources_desc = (
    f"Sadhguru:{len(sadhguru_texts)}, "
    f"Classical:{len(classical_texts)}, "
    f"Wiki:{len(wiki_texts)}, "
    f"ChatReplay:{len(chat_texts)}"
)

packed_dataset.push_to_hub(
    HF_DATASET,
    private=False,
    commit_message=(
        f"Clean DAPT v2.0: {len(blocks):,} blocks x {BLOCK_SIZE} tokens | "
        f"Sources: {sources_desc} | "
        f"Tamil >= {TAMIL_THRESHOLD:.0%} | NFKC cleaned | "
        f"Tokenizer: {MODEL_ID}"
    ),
)

print(f"\n\u2705 Dataset uploaded: https://huggingface.co/datasets/{HF_DATASET}")

# Verify upload
print(f"\n\U0001f50d Verifying upload...")
verify_ds = load_dataset(HF_DATASET)
print(f"   Blocks: {len(verify_ds['train']):,}")
sample = verify_ds['train'][0]
assert len(sample['input_ids']) == BLOCK_SIZE, "Block size mismatch!"
print(f"   \u2705 Verification passed")

📊 Dataset created:
   Blocks:       4,683
   Total tokens: 4,795,392
   Columns:      ['input_ids', 'attention_mask', 'labels']

📤 Uploading to CryptoYogi/vazhi-dapt-tamil-v2_0...


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

...


✅ Dataset uploaded: https://huggingface.co/datasets/CryptoYogi/vazhi-dapt-tamil-v2_0

🔍 Verifying upload...


README.md:   0%|          | 0.00/352 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/8.49M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4683 [00:00<?, ? examples/s]

   Blocks: 4,683
   ✅ Verification passed


In [29]:
# Cell 17 — Summary

print(f"{'='*60}")
print(f"\U0001f4cb CLEAN DAPT v2.0 DATA PREP — SUMMARY")
print(f"{'='*60}")
print(f"")
print(f"   Dataset:         {HF_DATASET}")
print(f"   Total blocks:    {len(blocks):,}")
print(f"   Block size:      {BLOCK_SIZE} tokens")
print(f"   Total tokens:    {total_tokens:,}")
print(f"   Tamil threshold: >= {TAMIL_THRESHOLD:.0%}")
print(f"")
print(f"   Sources:")
print(f"     Sadhguru articles:  {len(sadhguru_texts):>6,} docs (~{est_sadhguru_tokens:,} tokens)")
print(f"     Classical lit:      {len(classical_texts):>6,} docs (~{est_classical_tokens:,} tokens)")
if wiki_texts:
    print(f"     Wiki (IndicAlign): {len(wiki_texts):>6,} docs (~{wiki_stats.get('tokens', 0):,} tokens)")
print(f"     Chat replay:       {len(chat_texts):>6,} docs (~{chat_token_count:,} tokens, {chat_pct_of_total:.1%})")
print(f"")
print(f"   Cleaning: NFKC + article artifacts + dedup + Tamil gate")
print(f"")
print(f"\U0001f449 Next: Run Vazhi_DAPT_v2_0_Tamil.ipynb on Colab Pro (GPU)")
print(f'   It will load: ds = load_dataset(\"{HF_DATASET}\")')

📋 CLEAN DAPT v2.0 DATA PREP — SUMMARY

   Dataset:         CryptoYogi/vazhi-dapt-tamil-v2_0
   Total blocks:    4,683
   Block size:      1024 tokens
   Total tokens:    4,795,392
   Tamil threshold: >= 90%

   Sources:
     Sadhguru articles:     561 docs (~3,877,822 tokens)
     Classical lit:       1,844 docs (~359,635 tokens)
     Chat replay:          384 docs (~59,259 tokens, 1.4%)

   Cleaning: NFKC + article artifacts + dedup + Tamil gate

👉 Next: Run Vazhi_DAPT_v2_0_Tamil.ipynb on Colab Pro (GPU)
   It will load: ds = load_dataset("CryptoYogi/vazhi-dapt-tamil-v2_0")
